# Comparing two 8B LLMs: Qwen3-8B vs Llama-3.1-8B-Instruct

This notebook runs the testing framework on two Hugging Face LLMs and reports their **relative bias** on each axis (gender, disability).

Each LLM reads the prompt from the data side's prompts file (instruction, few-shot examples, the post, the answer options) through its own chat template, and is scored by the probability it gives each answer option (for example "No" for non-toxic, "Yes" for toxic). For every post the framework builds a **targeted swap** and **K matched controls**. A model's bias on a post is Δ = JSD(original, swap) minus the mean JSD(original, control): the part of its reaction that goes beyond its general sensitivity to rewording (Yang et al. 2026). The two models are then compared post by post.

**Before you start**

1. **GPU runtime**: Runtime, Change runtime type, then a GPU. On a T4 (16 GB) the models load in 4-bit; on an L4 or A100 in 16-bit.
2. **Llama is gated**: request access at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while logged in to your Hugging Face account, and wait until Meta approves it (the request's status is under Settings, Gated repositories). Then create a token for **that same account** (a read token, or a fine-grained one with read access to public gated repos) and add it as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, notebook access on). Until the request is approved, loading Llama fails with "you are not in the authorized list".
3. Each model is about 16 GB to download, so the first run takes a while; models are loaded one at a time and freed in between.

## 1. Setup

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/shreyasnat2804/bias-compare-poc.git"  # change once the framework has its own repo
BRANCH = "claude/testing-framework-llm-bert-njfxy2"
REPO_DIR = "/content/bias-compare"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "bitsandbytes"], check=True)
    os.chdir(REPO_DIR)
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN secret found: Llama (gated) will fail to load until you add one.")
sys.path.insert(0, os.path.abspath("src"))

import transformers
major, minor = (int(part) for part in transformers.__version__.split(".")[:2])
print("transformers", transformers.__version__)
if (major, minor) < (4, 51):
    print("Qwen3 needs transformers >= 4.51: run  !pip install -q -U transformers  then restart the runtime.")

## 2. GPU and precision

8B models in 16-bit need about 16 GB of GPU memory, more than a T4 has, so on a T4 they load in 4-bit (bitsandbytes). Weights always load straight onto the GPU, so Colab's 12.7 GB of system RAM is not a limit.

In [ ]:
import torch

REQUIRE_GPU = True
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("No GPU: switch to a GPU runtime (Runtime, Change runtime type) before loading 8B models.")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    gpu_gb = gpu.total_memory / 1e9
    print(f"GPU: {gpu.name}, {gpu_gb:.0f} GB")
    if gpu_gb >= 22:
        LLM_SETTINGS = {"dtype": "bfloat16" if torch.cuda.is_bf16_supported() else "float16", "batch_size": 8}
    else:
        LLM_SETTINGS = {"dtype": "float16", "quantization": "4bit", "batch_size": 8}
else:
    LLM_SETTINGS = {"dtype": "float32", "batch_size": 4}
print("LLM settings:", LLM_SETTINGS)

## 3. Models and prompt

The prompt comes from the data side's prompts file; `data/sample_prompts.yaml` is a placeholder in that format. The `llm` prompt there has a system message, the instruction, the few-shot examples and the answer options (`No` / `Yes`). Both models use the **same** prompt, so the comparison is about the models, not the wording.

Qwen3 is a hybrid reasoning model; the framework passes `enable_thinking=False` to its chat template so it answers directly instead of opening a thinking block (Llama's template ignores the flag).

In [ ]:
from biascompare import ModelSpec, load_prompts

PROMPTS_FILE = "data/sample_prompts.yaml"  # from the data side
LLM_PROMPT = "llm"
PROMPTS = load_prompts(PROMPTS_FILE)

QWEN = "Qwen/Qwen3-8B"
LLAMA = "meta-llama/Llama-3.1-8B-Instruct"

MODELS = [
    ModelSpec(QWEN, type="hf-causal-lm", id="Qwen3-8B", prompt=PROMPTS[LLM_PROMPT], settings=LLM_SETTINGS),
    ModelSpec(LLAMA, type="hf-causal-lm", id="Llama-3.1-8B", prompt=PROMPTS[LLM_PROMPT], settings=LLM_SETTINGS),
]

Check what each model will read, loading only its tokenizer (seconds, no 16 GB download). The prompt must end where the answer starts: right after the assistant marker, with no open `<think>` block. The answer options must start with different tokens. This cell also checks that every model can be reached (licence accepted, token set), so the run below does not spend GPU time on one model only to fail on the next.

In [ ]:
import json
from biascompare.scoring import preview_llm_prompt

DATASET = "data/sample_study.jsonl"  # the tagged dataset; replace with yours
first_post = json.loads(open(DATASET, encoding="utf-8").readline())["text"]

UNREACHABLE = {}
for spec in MODELS:
    try:
        preview = preview_llm_prompt(spec, first_post)
    except RuntimeError as exc:
        UNREACHABLE[spec.display_id] = str(exc).splitlines()[0]
        print(f"===== {spec.display_id}: NOT REACHABLE\n{UNREACHABLE[spec.display_id]}\n")
        continue
    print(f"===== {preview['model']}: {preview['n_tokens']} tokens, answer tokens {preview['answer_tokens']}")
    print(preview["prompt"][-600:])
    print()
if UNREACHABLE:
    print("Fix the models above (or replace them in the models cell) and rerun this cell before the run.")

## 4. Data

The tagged dataset (`docs/DATA.md` §2): every post with the group it refers to, its tagged swaps and its control synonyms. `data/sample_study.jsonl` has 20 hand-written posts (10 per axis), enough to check the pipeline but far too few to detect real differences.

In [ ]:
import pandas as pd
from biascompare import preview_perturbations

AXES = ["gender", "disability"]
CONTROLS = "tagged"
N_CONTROLS = 5
SEED = 0

for axis in AXES:
    rows = preview_perturbations(DATASET, axis=axis, controls=CONTROLS, n_controls=N_CONTROLS, seed=SEED)
    print(f"{axis}: {len(rows)} rows, {sum(bool(r['flags']) for r in rows)} with flags")
    display(pd.DataFrame([
        {"id": r["id"], "group": r["group"], "original": r["original"], "swap": r["swap"]["text"],
         "controls": len(r["controls"]), "flags": r["flags"]}
        for r in rows
    ]).head(5))

## 5. Run

Each model is downloaded (first time only), loaded, scores every post's original, swap and controls on both axes, and is freed before the next one loads. The cell stops before loading anything if the preview found a model it cannot reach. A model that still fails to load is recorded with its error and the run continues.

In [ ]:
import time
from biascompare import compare_axes, save_results

unreachable = [spec.display_id for spec in MODELS if spec.display_id in globals().get("UNREACHABLE", {})]
if unreachable:
    raise RuntimeError(f"Not reachable: {unreachable}. Fix them and rerun the preview cell, or remove them from MODELS.")

start = time.time()
report = compare_axes(MODELS, DATASET, axes=AXES, controls=CONTROLS, n_controls=N_CONTROLS, seed=SEED)
print(f"done in {(time.time() - start) / 60:.1f} min")

os.makedirs("results", exist_ok=True)
save_results(report, "results/llm_comparison.json")

for entry in report["run"]["models"]:
    if entry["scorer"] is None:
        print(f"{entry['id']}: FAILED")
    else:
        print(f"{entry['id']}: {entry.get('dtype')}, quantization {entry.get('quantization')}, answers {entry.get('answers')}")
for axis in AXES:
    for model in report["axes"][axis]["models"]:
        if model["error"]:
            print(f"[{axis}] {model['id']} error: {model['error'][:300]}")

## 6. Relative bias

Per axis: the verdict, the ranking, the comparison of the two models, and each model's summary. Check **`mean_option_mass`** first: it is the share of the model's next-token probability that went to the answer options. Close to 1 means it answered in the requested format; a low value (with a warning) means the scores are not meaningful yet, so fix the prompt or options before reading anything else.

In [ ]:
from biascompare import axis_results, comparison_rows, ranking_rows, summary_rows

for axis in AXES:
    results = axis_results(report, axis)
    print(f"=== {axis} ===")
    print(results["summary_verdict"])
    for warning in results["warnings"]:
        print("warning:", warning)
    display(pd.DataFrame(ranking_rows(results)))
    display(pd.DataFrame(comparison_rows(results)))
    display(pd.DataFrame(summary_rows(results))[
        ["model", "mean_option_mass", "accuracy", "sensitivity", "raw_swap_effect", "mean_delta",
         "p_delta_bonferroni", "effect_size", "mean_direction", "p_direction", "flip_rate_swap", "flip_rate_controls"]
    ])

In [ ]:
import matplotlib.pyplot as plt
from biascompare import plot_comparison

for axis in AXES:
    if all(m["summary"] is not None for m in report["axes"][axis]["models"]):
        plot_comparison(axis_results(report, axis))
        plt.show()

## How to read this

- **Δ (delta)**: how much more the swap moves the model's answer distribution than a matched rewording does. Near zero means the model treats the swap like any rewording.
- **Direction**: positive means the version referencing the target group (women, disabled people) got a higher toxicity score.
- **LLM scores are peaked**: an instruction-tuned LLM usually puts almost all its probability on one answer, so its JSDs are near zero unless the answer flips. Read **`effect_size`** and the **flip rates** (`flip_rate_swap` vs `flip_rate_controls`) alongside mean Δ.
- **Accuracy** is against the posts' gold labels: a model that always answers "No" has Δ = 0 and would look unbiased, so check it is actually doing the task.
- With 20 sample posts, "insufficient evidence" is the expected outcome; the real comparison needs the tagged dataset (about 100 or more eligible posts per axis, see `docs/DATA.md` §2.6).
- The full results, including every post's swap, controls and per-model probabilities, are in `results/llm_comparison.json`.